# Tome 11, Chapter 5: Brownian motion and analysis (Python companion)

Exercises first, solutions at the end. The exit time of a ball, the Cauchy law hiding in the half-plane, the Cameron–Martin $\cosh$ formula checked by simulation, Girsanov's theorem turned into an algorithm (importance sampling of a rare event), the annulus formula behind recurrence and transience, and a computer science application: **Muller's walk on spheres**, a mesh-free Monte Carlo solver for the Dirichlet problem, still used in graphics and engineering. Requires `numpy`.

In [1]:
import math
import numpy as np
rng = np.random.default_rng(1105)

def sphere(n, d):
    # n independent uniform points on the unit sphere of R^d (Lemma 5.7)
    v = rng.standard_normal((n, d))
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def walk_on_spheres(x0, dist, m, eps):
    # m walks from x0; dist(Y) = distance from each row of Y to the boundary.
    # Returns the stopping points (within eps of the boundary) and the step counts.
    X = np.tile(np.asarray(x0, float), (m, 1)); steps = np.zeros(m, int)
    alive = np.ones(m, bool)
    while alive.any():
        idx = np.where(alive)[0]
        r = dist(X[idx]); stop = r < eps
        alive[idx[stop]] = False
        go = idx[~stop]
        X[go] += r[~stop, None] * sphere(len(go), X.shape[1]); steps[go] += 1
    return X, steps

## Exercises

**P5.1 (★) The exit time of a ball.** Simulate Brownian motion from $x=(\frac12,0,\dots,0)$ in $\mathbb R^d$ with time step $\Delta t$ until it leaves the unit ball, for $d=1,2,3$ and $\Delta t=10^{-2},10^{-3}$, and compare the mean exit time with $\frac{1-|x|^2}d$ (Exercise 5.1). The discrete walk overshoots: how does the bias depend on $\Delta t$?

**P5.2 (★) A Cauchy law in the half-plane.** Planar Brownian motion from $(0,1)$ is stopped on the real axis. Using the law of the hitting time of a level (Chapter 2), sample the hitting point exactly, compare its quantiles with those of the Cauchy law (Exercise 5.2), and watch the running mean of the samples refuse to settle.

**P5.3 (★★) The $\cosh$ formula.** Check Example 5.3, $\mathbb E\exp(-\frac{\lambda^2}2\int_0^1B_s^2\,\mathrm ds)=(\cosh\lambda)^{-1/2}$, by Monte Carlo for $\lambda=\frac12,1,2,4$.

**P5.4 (★★) Girsanov as an algorithm.** Estimate $p=P(B_1>5)\approx2.87\times10^{-7}$. The naive estimator sees nothing. Instead sample $B$ with drift $\mu$, that is $B_1\sim N(\mu,1)$, and reweight by the Cameron–Martin density $e^{-\mu B_1+\mu^2/2}$ (Corollary 5.6). Compare the relative error for $\mu=3,5,6$. Which $\mu$ is best, and why?

**P5.5 (★★) The annulus formula.** Brownian motion from $|x|=\frac12$ in the annulus $\frac1{10}<|y|<1$: estimate the probability that it reaches the inner circle first, in $d=2$ and $d=3$, and compare with Theorem 5.12. Use the walk on spheres of P5.6 (the distance to the boundary is $\min(|y|-r,R-|y|)$).

**P5.6 (★★★) CS application: the walk on spheres.** Kakutani's theorem $u(x)=\mathbb E_x\,g(B_\tau)$ (Theorem 5.10) needs only the exit point, not the path. Since Brownian motion from the centre of a ball leaves it at a uniform point (Lemma 5.9 and the strong Markov property), we may jump from $x$ to a uniform point on the largest sphere around $x$ inside $D$, and repeat until we are within $\varepsilon$ of $\partial D$ (M. E. Muller, 1956). Solve the Dirichlet problem on the unit square with $g(x,y)=x^3-3xy^2$ (harmonic, so $u=g$) at the point $(0.3,0.6)$, for $\varepsilon=10^{-1},\dots,10^{-6}$. How do the error and the number of jumps depend on $\varepsilon$?

## Solutions

### P5.1

**How to find it.** Run many walkers in parallel, advancing only those still inside the ball. A discrete walk can only notice the exit after the fact, at a grid time, so it leaves late: each crossing is detected about one step too late, and one step of Brownian motion has size $\sqrt{\Delta t}$.

In [2]:
def exit_time(x, R, dt, m):
    X = np.tile(np.asarray(x, float), (m, 1)); t = np.zeros(m); alive = np.ones(m, bool)
    while alive.any():
        X[alive] += rng.standard_normal((alive.sum(), len(x))) * math.sqrt(dt)
        t[alive] += dt
        alive &= (X**2).sum(1) < R * R
    return t

m = 4000
for d in (1, 2, 3):
    x = [0.5] + [0] * (d - 1)
    for dt in (1e-2, 1e-3):
        t = exit_time(x, 1.0, dt, m)
        print(f"d = {d}, dt = {dt:<6}: mean = {t.mean():.4f} +- {t.std() / math.sqrt(m):.4f}"
              f"   exact {0.75 / d:.4f}   bias {t.mean() - 0.75 / d:+.4f}")

d = 1, dt = 0.01  : mean = 0.8661 +- 0.0139   exact 0.7500   bias +0.1161


d = 1, dt = 0.001 : mean = 0.7940 +- 0.0128   exact 0.7500   bias +0.0440
d = 2, dt = 0.01  : mean = 0.4361 +- 0.0060   exact 0.3750   bias +0.0611


d = 2, dt = 0.001 : mean = 0.3950 +- 0.0057   exact 0.3750   bias +0.0200
d = 3, dt = 0.01  : mean = 0.3035 +- 0.0038   exact 0.2500   bias +0.0535


d = 3, dt = 0.001 : mean = 0.2678 +- 0.0034   exact 0.2500   bias +0.0178


The estimates are biased upwards, and the bias drops by a factor of about $3\approx\sqrt{10}$ when $\Delta t$ is divided by $10$: the error of the naive scheme is of order $\sqrt{\Delta t}$, not $\Delta t$. This is the price of detecting a continuous exit on a grid (Gobet, *Weak approximation of killed diffusion using Euler schemes*, Stochastic Process. Appl. 87, 2000, proves the $\sqrt{\Delta t}$ rate; shifting the boundary inwards by $0.5826\sqrt{\Delta t}$ removes its leading term, the continuity correction of Broadie, Glasserman and Kou, Math. Finance 7, 1997). The walk on spheres of P5.6 avoids the grid altogether.

### P5.2

**How to find it.** The vertical coordinate $1+W_t$ hits $0$ at $T=1/Z^2$ with $Z\sim N(0,1)$ (Chapter 2: $T_a\overset{d}{=}a^2/Z^2$). The horizontal coordinate is an independent Brownian motion, so the hitting point is $\sqrt T\,Z'$ with $Z'$ independent: the ratio $Z'/|Z|$ of two Gaussians. The Cauchy quantile of order $q$ is $\tan(\pi(q-\frac12))$.

In [3]:
m = 200_000
T = 1 / rng.standard_normal(m)**2
X = np.sqrt(T) * rng.standard_normal(m)
for q in (0.6, 0.75, 0.9, 0.99):
    print(f"q = {q}: empirical {np.quantile(X, q):8.3f}   Cauchy {math.tan(math.pi * (q - 0.5)):8.3f}")
print("running means:", [round(float(X[:n].mean()), 2) for n in (10**3, 10**4, 10**5, 2 * 10**5)])

q = 0.6: empirical    0.318   Cauchy    0.325
q = 0.75: empirical    0.993   Cauchy    1.000
q = 0.9: empirical    3.053   Cauchy    3.078
q = 0.99: empirical   30.131   Cauchy   31.821
running means: [0.65, 0.34, 0.46, -1.53]


The quantiles agree to about two decimals; the $99\%$ quantile is noisier (about $30$ against $31.8$), since only $1\%$ of the samples lie beyond it. The running means wander: the Cauchy law has no mean, the law of large numbers does not apply, and the average of $n$ samples is again Cauchy, however large $n$ is. A Brownian particle released at height $1$ above a wall lands, with probability $1\%$, more than $63$ units away.

### P5.3

**How to find it.** Simulate the paths on a grid of $1000$ steps, integrate $B^2$ by the trapezoid rule, and average $e^{-\lambda^2S/2}$, which lies in $[0,1]$, so the Monte Carlo error is easy to control.

In [4]:
N, m = 1000, 20_000
dt = 1 / N
for lam in (0.5, 1, 2, 4):
    S = np.zeros(m); B = np.zeros(m)
    for k in range(N):
        Bn = B + rng.standard_normal(m) * math.sqrt(dt)
        S += 0.5 * (B**2 + Bn**2) * dt; B = Bn
    v = np.exp(-lam**2 / 2 * S)
    print(f"lambda = {lam}: Monte Carlo {v.mean():.4f} +- {v.std() / math.sqrt(m):.4f}"
          f"   (cosh lambda)^(-1/2) = {1 / math.sqrt(math.cosh(lam)):.4f}")

lambda = 0.5: Monte Carlo 0.9418 +- 0.0004   (cosh lambda)^(-1/2) = 0.9417


lambda = 1: Monte Carlo 0.8065 +- 0.0012   (cosh lambda)^(-1/2) = 0.8050


lambda = 2: Monte Carlo 0.5120 +- 0.0020   (cosh lambda)^(-1/2) = 0.5156


lambda = 4: Monte Carlo 0.1901 +- 0.0015   (cosh lambda)^(-1/2) = 0.1914


All four agree within two standard errors. Each $\lambda$ uses fresh paths, and the formula, found by Cameron and Martin through a Gaussian ansatz in a partial differential equation, is confirmed by nothing more than coin flips.

### P5.4

**How to find it.** Under $Q$, $B_1\sim N(\mu,1)$, and $\frac{\mathrm dP}{\mathrm dQ}=e^{-\mu B_1+\mu^2/2}$. The estimator $\frac1n\sum e^{-\mu Y_i+\mu^2/2}\mathbf 1_{\{Y_i>5\}}$, $Y_i\sim N(\mu,1)$, is unbiased for every $\mu$; its variance depends on $\mu$. Report the relative standard error, standard deviation divided by $p\sqrt n$.

In [5]:
p = 0.5 * math.erfc(5 / math.sqrt(2))
print(f"exact p = {p:.4e}")
for n in (10**4, 10**6):
    print(f"n = {n}: naive estimate {np.mean(rng.standard_normal(n) > 5):.1e}")
    for mu in (3, 5, 6):
        Y = rng.standard_normal(n) + mu
        w = np.exp(-mu * Y + mu * mu / 2) * (Y > 5)
        print(f"   mu = {mu}: estimate {w.mean():.4e}, relative error {w.std() / math.sqrt(n) / p:.4f}")

exact p = 2.8665e-07
n = 10000: naive estimate 0.0e+00
   mu = 3: estimate 2.8307e-07, relative error 0.0773
   mu = 5: estimate 2.8779e-07, relative error 0.0239
   mu = 6: estimate 3.0183e-07, relative error 0.0314
n = 1000000: naive estimate 1.0e-06
   mu = 3: estimate 2.8572e-07, relative error 0.0078
   mu = 5: estimate 2.8794e-07, relative error 0.0024
   mu = 6: estimate 2.8748e-07, relative error 0.0030


With $10^4$ samples the naive method returns $0$, and with $10^6$ it sees a single hit or none; the reweighted one already has a relative error of a few percent, and $\mu=5$ is the best of the three. Tilting the drift to the level of the rare event makes it typical, and the weights stay moderate; a smaller $\mu$ hits $\{B_1>5\}$ too rarely, a larger one pays with tiny, wildly varying weights. That $\mu=a$ is asymptotically optimal for $P(B_1>a)$ as $a\to\infty$ is a classical result of rare-event simulation (Siegmund, *Importance sampling in the Monte Carlo study of sequential tests*, Ann. Statist. 4, 1976; Asmussen and Glynn, *Stochastic Simulation*, Springer, 2007, Chapter VI).

### P5.5

**How to find it.** Run the walk on spheres in the annulus with a small $\varepsilon$; a walk stops near the inner circle when $|y|$ is closer to $r$ than to $R$. Theorem 5.12 gives $\frac{\log(R/|x|)}{\log(R/r)}$ in the plane and $\frac{|x|^{2-d}-R^{2-d}}{r^{2-d}-R^{2-d}}$ for $d\ge3$.

In [6]:
r, R, x = 0.1, 1.0, 0.5
m = 20_000
for d in (2, 3):
    dist = lambda Y: np.minimum(np.linalg.norm(Y, axis=1) - r, R - np.linalg.norm(Y, axis=1))
    X, steps = walk_on_spheres([x] + [0] * (d - 1), dist, m, 1e-4)
    est = np.mean(np.linalg.norm(X, axis=1) < (r + R) / 2)
    exact = math.log(R / x) / math.log(R / r) if d == 2 else (x**(2 - d) - R**(2 - d)) / (r**(2 - d) - R**(2 - d))
    print(f"d = {d}: estimate {est:.4f} +- {math.sqrt(est * (1 - est) / m):.4f}   exact {exact:.4f}"
          f"   mean number of jumps {steps.mean():.1f}")

d = 2: estimate 0.2991 +- 0.0032   exact 0.3010   mean number of jumps 14.1
d = 3: estimate 0.1094 +- 0.0022   exact 0.1111   mean number of jumps 27.7


Both agree within the Monte Carlo error. In the plane the chance of reaching the small disc is $\log2/\log10\approx0.30$, and it would tend to $1$ as $R\to\infty$, only very slowly (recurrence); in space it is $\frac19$, and it would tend to $|x|^{-1}r=\frac15$ (transience).

### P5.6

**How to find it.** For the unit square the distance to the boundary is $\min(x,1-x,y,1-y)$. Stop when it is below $\varepsilon$ and score $g$ at the stopping point. Two sources of error: the Monte Carlo error, of order $\sigma/\sqrt m$ whatever $\varepsilon$, and the bias from stopping at distance $\varepsilon$ instead of on $\partial D$, of order $\varepsilon$ since $g$ is smooth.

In [7]:
g = lambda Y: Y[:, 0]**3 - 3 * Y[:, 0] * Y[:, 1]**2
dist = lambda Y: np.minimum.reduce([Y[:, 0], 1 - Y[:, 0], Y[:, 1], 1 - Y[:, 1]])
x0 = np.array([0.3, 0.6]); u = x0[0]**3 - 3 * x0[0] * x0[1]**2
m = 40_000
print(f"exact u(x0) = {u:.4f}")
for eps in (1e-1, 1e-2, 1e-3, 1e-4, 1e-6):
    X, steps = walk_on_spheres(x0, dist, m, eps)
    v = g(X)
    print(f"eps = {eps:<6}: error {v.mean() - u:+.4f} (standard error {v.std() / math.sqrt(m):.4f})"
          f"   mean number of jumps {steps.mean():.1f}")

exact u(x0) = -0.2970
eps = 0.1   : error -0.0007 (standard error 0.0028)   mean number of jumps 2.8
eps = 0.01  : error -0.0025 (standard error 0.0031)   mean number of jumps 6.1
eps = 0.001 : error +0.0043 (standard error 0.0032)   mean number of jumps 9.4


eps = 0.0001: error -0.0041 (standard error 0.0032)   mean number of jumps 12.7


eps = 1e-06 : error +0.0030 (standard error 0.0032)   mean number of jumps 19.2


Already at $\varepsilon=10^{-2}$ the bias is lost in the Monte Carlo noise, and the number of jumps grows only like $\log(1/\varepsilon)$: about three more jumps for each factor of $10$. A walk crosses the square in a handful of giant leaps, where a Brownian path on a grid would need thousands of small steps (compare P5.1). And no mesh is needed: the method evaluates $u$ at a single point, on any domain for which one can compute the distance to the boundary, which is why it has been revived in computer graphics for geometry processing on complicated shapes (Sawhney and Crane, *Monte Carlo geometry processing: a grid-free approach to PDE-based methods on volumetric domains*, ACM Trans. Graph. 39(4), 2020). The method itself is due to M. E. Muller, *Some continuous Monte Carlo methods for the Dirichlet problem*, Ann. Math. Statist. 27 (1956), 569–589.